In [ ]:
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
import numpy as np
import pandas as pd

# Load the geojson data
gdf = gpd.read_file('../data/2023/311_Encampment_Reports%2C_City_of_San_Diego%2C_2023.geojson')


In [ ]:
# Filter out rows where geometry is None
gdf = gdf[gdf.geometry.notnull()]

In [ ]:

# # Extract geometry (points) and convert them to numpy array for kernel density estimation
# coords = np.array([[point.x, point.y] for point in gdf.geometry])

# # Perform Kernel Density Estimation (KDE)
# kde = KernelDensity(bandwidth=0.005, kernel='gaussian')
# kde.fit(coords)

# # Create a grid to evaluate the KDE
# x_min, y_min = coords.min(axis=0)
# x_max, y_max = coords.max(axis=0)

# x_grid = np.linspace(x_min, x_max, 500)
# y_grid = np.linspace(y_min, y_max, 500)
# x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
# xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T

# # Evaluate the KDE on this grid
# z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)

# # 1. Plot with only point data overlaid on the map
# fig, ax = plt.subplots(figsize=(10, 10))

# # Plot the encampment locations
# gdf.plot(ax=ax, marker='o', color='red', markersize=2, label="Encampment Locations")

# # Add the basemap
# ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

# plt.title("Point Data of Encampment Reports (2023) Overlaid on San Diego County Map")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.legend()
# plt.show()


In [ ]:

# # 2. Plot with only the kernel density heatmap overlaid on the map
# fig, ax = plt.subplots(figsize=(10, 10))

# # Plot the kernel density estimation map
# ax.imshow(z, origin='lower', cmap='inferno', extent=[x_min, x_max, y_min, y_max], alpha=1)

# # Add the basemap
# ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4)

# plt.title("Kernel Density Heatmap of Encampment Reports (2023) Overlaid on San Diego County Map")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.show()


## Okay now Kaelan. CMON use your fkcn brain here. You need to break this down by different time periods. and also asjust different parameters of the kernel density calculations

In [ ]:
gdf.head()

In [ ]:
# Assuming your date column is named 'date'
# Get min and max dates
min_date = gdf['date_requested'].min()
max_date = gdf['date_requested'].max()

# If your dates aren't already in datetime format, convert first:
gdf['date_requested'] = pd.to_datetime(gdf['date_requested'])
min_date = gdf['date_requested'].min()
max_date = gdf['date_requested'].max()

# To get the total date range (number of days)
date_range = (max_date - min_date).days

print(f"Date range: {min_date} to {max_date}")
print(f"Total days: {date_range}")

In [ ]:
gdf.columns

In [ ]:
gdf.shape

In [ ]:
gdf['date_requested'] = pd.to_datetime(gdf['date_requested'])


In [ ]:
gdf['year_month'] = gdf['date_requested'].dt.to_period('M')
gdf['year_week'] = gdf['date_requested'].dt.to_period('W')
gdf['year_day'] = gdf['date_requested'].dt.to_period('D')


In [ ]:
gdf.head()

In [ ]:
# Redefine the seasons based on months
spring_months = [3, 4, 5]
summer_months = [6, 7, 8]
fall_months = [9, 10, 11]
winter_months = [12, 1, 2]

# Extract the month from the 'date_reque' column
gdf['month'] = gdf['date_requested'].dt.month

# Create separate dataframes for each season
spring_df = gdf[gdf['month'].isin(spring_months)]
summer_df = gdf[gdf['month'].isin(summer_months)]
fall_df = gdf[gdf['month'].isin(fall_months)]
winter_df = gdf[gdf['month'].isin(winter_months)]

In [ ]:
summer_df.shape

In [ ]:
# Function to assign seasons based on the month
def assign_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'

# Apply the function to the 'month' column
gdf['season'] = gdf['month'].apply(assign_season)

# Plot the distribution of data points by season
season_distribution = gdf['season'].value_counts().reindex(['Winter', 'Spring', 'Summer', 'Fall'])

fig, ax = plt.subplots(figsize=(10, 6))
season_distribution.plot(kind='bar', color='lightcoral', ax=ax)
ax.set_title('Distribution of Data Points by Season 2023 (Winter, Spring, Summer, Fall)')
ax.set_xlabel('Season')
ax.set_ylabel('Number of Data Points')
plt.show()


In [ ]:
# Create individual dataframes for each month
jan_df = gdf[gdf['month'] == 1]
feb_df = gdf[gdf['month'] == 2]
mar_df = gdf[gdf['month'] == 3]
apr_df = gdf[gdf['month'] == 4]
may_df = gdf[gdf['month'] == 5]
jun_df = gdf[gdf['month'] == 6]
jul_df = gdf[gdf['month'] == 7]
aug_df = gdf[gdf['month'] == 8]
sep_df = gdf[gdf['month'] == 9]
oct_df = gdf[gdf['month'] == 10]
nov_df = gdf[gdf['month'] == 11]
dec_df = gdf[gdf['month'] == 12]

# Confirm that the dataframes were created
(jan_df.shape, feb_df.shape, mar_df.shape, apr_df.shape, may_df.shape, jun_df.shape,
 jul_df.shape, aug_df.shape, sep_df.shape, oct_df.shape, nov_df.shape, dec_df.shape)


In [ ]:
# Create a dictionary of all months with their respective number of data points
all_month_data = {
    'January': jan_df.shape[0],
    'February': feb_df.shape[0],
    'March': mar_df.shape[0],
    'April': apr_df.shape[0],
    'May': may_df.shape[0],
    'June': jun_df.shape[0],
    'July': jul_df.shape[0],
    'August': aug_df.shape[0],
    'September': sep_df.shape[0],
    'October': oct_df.shape[0],
    'November': nov_df.shape[0],
    'December': dec_df.shape[0]
}

# Create a bar plot to show the distribution of points across all months
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(all_month_data.keys(), all_month_data.values(), color='cornflowerblue')
ax.set_title('Distribution of Data Points Across All Months (2023)')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Data Points')
ax.set_xticklabels(all_month_data.keys(), rotation=45)
plt.show()


In [ ]:
# Calculate the global extent (bounding box)
x_min_global = gdf.geometry.x.min()
x_max_global = gdf.geometry.x.max()
y_min_global = gdf.geometry.y.min()
y_max_global = gdf.geometry.y.max()

In [ ]:
def plot_kernel_density_city_center(gdf, title, center_lon, center_lat, lon_range=0.05, lat_range=0.05, bandwidth=0.005):
    # Ensure geometries are valid
    gdf = gdf[gdf.geometry.notnull()]
    
    if gdf.empty:
        print(f"Skipping {title} as there are no data points.")
        return
    
    # Extract geometry points
    coords = np.array([[point.x, point.y] for point in gdf.geometry])
    
    if coords.shape[0] == 0:
        print(f"Skipping {title} due to no coordinates.")
        return
    
    # Perform Kernel Density Estimation (KDE)
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(coords)
    
    # Create a grid to evaluate the KDE
    x_min_zoomed = center_lon - lon_range / 2
    x_max_zoomed = center_lon + lon_range / 2
    y_min_zoomed = center_lat - lat_range / 2
    y_max_zoomed = center_lat + lat_range / 2
    
    x_grid = np.linspace(x_min_zoomed, x_max_zoomed, 500)
    y_grid = np.linspace(y_min_zoomed, y_max_zoomed, 500)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
    xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T
    z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)

    # Plot the KDE with a basemap and the zoomed-in extent around the city center
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(z, origin='lower', cmap='inferno', extent=[x_min_zoomed, x_max_zoomed, y_min_zoomed, y_max_zoomed], alpha=1)
    
    # Add basemap
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4)
    
    plt.title(title)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()


In [ ]:
# Example: Zoom into the city center of San Diego (approx. longitude: -117.1611, latitude: 32.7157)
plot_kernel_density_city_center(
    gdf, 
    "2021", 
    center_lon=-117.1611,  # Approx. longitude of city center
    center_lat=32.7157,    # Approx. latitude of city center
    lon_range=0.05,        # Adjust range to zoom in/out
    lat_range=0.05         # Adjust range to zoom in/out
)


# shit ain't working, won't zoom, bad bad bad

In [ ]:
# # Calculate the global extent (bounding box)
# x_min_global = gdf.geometry.x.min()
# x_max_global = gdf.geometry.x.max()
# y_min_global = gdf.geometry.y.min()
# y_max_global = gdf.geometry.y.max()

# # Add padding to zoom in slightly and avoid maps being too zoomed out
# padding = 0.01  # Adjust this value as needed

# x_min_global -= padding
# x_max_global += padding
# y_min_global -= padding
# y_max_global += padding


In [ ]:
# Example: Zooming in by 50% (you can adjust the zoom factor)
x_min_zoomed, x_max_zoomed, y_min_zoomed, y_max_zoomed = zoom_in_extent(x_min_global, x_max_global, y_min_global, y_max_global, zoom_factor=0.5)

# Call the plot function with the zoomed-in extent
plot_kernel_density(winter_df, "2018", x_min_zoomed, x_max_zoomed, y_min_zoomed, y_max_zoomed)


In [ ]:
# Loop to generate monthly kernel density maps for 2023
for month in range(1, 13):
    month_df = gdf[(gdf['month'] == month) & (gdf['date_requested'].dt.year == 2023)]
    month_name = pd.to_datetime(f'2023-{month}-01').strftime('%B')
    title = f"{month_name} 2023 Kernel Density Map"
    plot_kernel_density(month_df, title)

In [ ]:
# Loop to generate seasonal kernel density maps for 2023
for season in ['Winter', 'Spring', 'Summer', 'Fall']:
    # Filter the data for the current season and year 2023
    season_df = gdf[(gdf['season'] == season) & (gdf['date_requested'].dt.year == 2023)]
    
    # Create the title for the plot
    title = f"{season} 2023 Kernel Density Map"
    
    # Plot the kernel density map for this season
    plot_kernel_density(season_df, title)

In [ ]:
# Filter data for August 2023 (Month = 8)
gdf['date_requested'] = pd.to_datetime(gdf['date_requested'])

aug_df = gdf[(gdf['month'] == 8) & (gdf['date_requested'].dt.year == 2023)]
gdf['week'] = gdf['date_requested'].dt.isocalendar().week


# Find the unique weeks in August
unique_weeks_in_aug = aug_df['week'].unique()

In [ ]:
# Loop to generate weekly kernel density maps for each week in July 2023
for week in unique_weeks_in_aug:
    # Filter data for the specific week in August 2023
    week_df = aug_df[aug_df['week'] == week]
    
    # Create the title for the plot
    title = f"Week {week}, December 2023 Kernel Density Map"
    
    # Plot the kernel density map for this week
    plot_kernel_density(week_df, title)